# core

> YouTube Transcript to structured Table of Contents

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pydantic import BaseModel, Field
from datetime import datetime
from typing import Literal

class Segment(BaseModel):
    "One parsed xscript segment (in-memory)."
    start: float = Field(ge=0, description="Start time in seconds")
    end: float = Field(ge=0, description="End time in seconds")
    text: str = Field(description="Normalized cue text")


class NormalizedSection(BaseModel):
    "One TOC section after normalization (path and end added to raw LLM output)."
    path: str = Field(description="Section path like '1', '2', ...")
    title: str = Field(description="Concise English section title")
    start: int = Field(ge=0, description="Start time in integer seconds")
    end: int = Field(ge=0, description="End time in integer seconds")

class Meta(BaseModel):
    "Cached video metadata (one per cached video; persisted as meta.json)."
    id: str = Field(description="YouTube video ID")
    title: str = Field(description="Video title")
    channel: str = Field(description="Channel name")
    duration: int = Field(ge=0, description="Duration in seconds")
    upload_date: str = Field(description="Upload date in YYYYMMDD format")
    webpage_url: str = Field(description="Canonical YouTube URL")
    description: str = Field(default='', description="Video description (may be empty)")
    captions: dict[str, Literal["auto", "manual"]] = Field(
        description="Caption availability map: lang code → caption type")
    last_used_at: datetime = Field(description="Last cache access time (UTC)")

def fmt_duration(seconds: int # Duration in seconds
                ) -> str: # Formatted as H:MM:SS or M:SS
    "Format seconds as human-readable duration."
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m}:{s:02d}'

def format_header(meta: "Meta | VideoBlock" # Meta or VideoBlock (both have title/channel/duration/upload_date)
                 ) -> str: # Formatted header string
    "Shared header for toc/sum/raw CLI commands."
    dur = fmt_duration(meta.duration)
    return f'# {meta.title}\nChannel: {meta.channel} | Duration: {dur} | {meta.upload_date}'

def slice_segments(segments: list[Segment], # List of Segment
                   start: int, # Section start in seconds
                   end: int # Section end in seconds
                  ) -> list[Segment]: # Segments within [start, end)
    "Return segments with start time inside [start, end)."
    return [s for s in segments if s.start >= start and s.start < end]

def format_toc_line(section: NormalizedSection, # NormalizedSection or subclass (AssembledSection, FlattenedSection)
                    url: str = '' # webpage_url for &t= deep link (omit when empty)
                   ) -> str: # Formatted line
    "Single-line TOC row for a section, optionally with deep-link URL."
    s_start = fmt_duration(section.start)
    s_end = fmt_duration(section.end)
    span = fmt_duration(section.end - section.start)
    suffix = f" {url}&t={section.start}" if url else ''
    return f"{section.path}. {section.title} {s_start}-{s_end} ({span}){suffix}"


In [ ]:
# Test: fmt_duration
assert fmt_duration(3991) == '1:06:31'
assert fmt_duration(195) == '3:15'
assert fmt_duration(0) == '0:00'
assert fmt_duration(60) == '1:00'
assert fmt_duration(3600) == '1:00:00'
print('ok')

In [ ]:
# Test: format_header
from datetime import datetime, timezone
hdr = format_header(Meta(
    id='T', title='Test Video', channel='Test Channel',
    duration=3991, upload_date='20260320',
    webpage_url='https://youtube.com/watch?v=T',
    description='', captions={'en': 'auto'},
    last_used_at=datetime(2026, 1, 1, tzinfo=timezone.utc)))
assert hdr == '# Test Video\nChannel: Test Channel | Duration: 1:06:31 | 20260320'
print('ok')

In [ ]:
# Test: format_toc_line
from yttoc.core import NormalizedSection
sec = NormalizedSection(path='1', title='Intro', start=0, end=137)
url = 'https://www.youtube.com/watch?v=ABC'
assert format_toc_line(sec, url) == '1. Intro 0:00-2:17 (2:17) https://www.youtube.com/watch?v=ABC&t=0'

sec2 = NormalizedSection(path='3', title='Deep dive', start=600, end=4191)
assert format_toc_line(sec2, url) == '3. Deep dive 10:00-1:09:51 (59:51) https://www.youtube.com/watch?v=ABC&t=600'

# Empty URL: no trailing space, no &t=
assert format_toc_line(sec) == '1. Intro 0:00-2:17 (2:17)'
print('ok')


In [ ]:
# Test: Segment validates non-negative timestamps
from yttoc.core import Segment
from pydantic import ValidationError

# Valid construction succeeds
s = Segment(start=0.0, end=1.5, text='x')
assert s.start == 0.0 and s.end == 1.5 and s.text == 'x'

# Negative start rejected
try:
    Segment(start=-0.001, end=0.0, text='x')
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for negative start'

# Negative end rejected
try:
    Segment(start=0.0, end=-1, text='x')
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for negative end'

print('ok')


In [ ]:
# Test: NormalizedSection validates required fields and non-negative bounds
from yttoc.core import NormalizedSection
from pydantic import ValidationError

# Valid construction succeeds
s = NormalizedSection(path='1', title='Intro', start=0, end=300)
assert s.path == '1' and s.title == 'Intro' and s.start == 0 and s.end == 300

# Negative start rejected
try:
    NormalizedSection(path='1', title='x', start=-1, end=10)
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for negative start'

# Negative end rejected
try:
    NormalizedSection(path='1', title='x', start=0, end=-1)
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for negative end'

# Missing required field rejected
try:
    NormalizedSection(path='1', title='x', start=0)  # no end
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for missing end'

print('ok')

In [ ]:
# Test: Meta validates required fields, captions Literal, last_used_at datetime
from yttoc.core import Meta
from pydantic import ValidationError
from datetime import datetime, timezone

# Valid construction succeeds (all 9 fields)
m = Meta(id='vid1', title='T', channel='Ch', duration=600,
         upload_date='20260101', webpage_url='https://youtube.com/watch?v=vid1',
         captions={'en': 'auto'},
         last_used_at=datetime(2026, 4, 16, 15, 13, 50, 653895, tzinfo=timezone.utc))
assert m.id == 'vid1'
assert m.description == ''
assert isinstance(m.last_used_at, datetime)

# Missing required field rejected (no channel)
try:
    Meta(id='x', title='t', duration=60, upload_date='20260101',
         webpage_url='u', captions={'en': 'auto'},
         last_used_at=datetime.now(timezone.utc))
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for missing channel'

# Negative duration rejected
try:
    Meta(id='x', title='t', channel='c', duration=-1, upload_date='20260101',
         webpage_url='u', captions={'en': 'auto'},
         last_used_at=datetime.now(timezone.utc))
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for negative duration'

# Invalid captions value rejected
try:
    Meta(id='x', title='t', channel='c', duration=60, upload_date='20260101',
         webpage_url='u', captions={'en': 'autop'},
         last_used_at=datetime.now(timezone.utc))
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for invalid caption type'

# Invalid last_used_at string rejected
try:
    Meta.model_validate_json(
        '{"id":"x","title":"t","channel":"c","duration":60,"upload_date":"20260101",'
        '"webpage_url":"u","captions":{"en":"auto"},"last_used_at":"yesterday"}'
    )
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for invalid last_used_at'

# Valid ISO last_used_at parses to datetime
m2 = Meta.model_validate_json(
    '{"id":"x","title":"t","channel":"c","duration":60,"upload_date":"20260101",'
    '"webpage_url":"u","captions":{"en":"auto"},'
    '"last_used_at":"2026-04-16T15:13:50.653895+00:00"}'
)
assert isinstance(m2.last_used_at, datetime)
assert m2.last_used_at.tzinfo is not None

print('ok')

In [ ]:
# Test: format_toc_line and format_header accept all typed inputs via polymorphism
from datetime import datetime, timezone
from yttoc.core import (Meta, NormalizedSection, format_header, format_toc_line)
from yttoc.summarize import VideoBlock, AssembledSection
from yttoc.map import FlattenedSection

# format_toc_line accepts NormalizedSection, AssembledSection, FlattenedSection
ns = NormalizedSection(path='1', title='Intro', start=0, end=300)
as_ = AssembledSection(path='1', title='Intro', start=0, end=300,
                        summary='s', keywords=['k'],
                        evidence={'text': 'e', 'at': 0})
fs = FlattenedSection(path='1', title='Intro', start=0, end=300,
                      summary='s', keywords=['k'], evidence={'text': 'e', 'at': 0},
                      lesson=1, video_id='X', video_title='V', jump_url='u')

line_ns = format_toc_line(ns, url='https://y.com/X')
line_as = format_toc_line(as_, url='https://y.com/X')
line_fs = format_toc_line(fs, url='https://y.com/X')
# All three produce the same line because only NormalizedSection fields are used
assert line_ns == line_as == line_fs
assert '1. Intro' in line_ns and '&t=0' in line_ns

# format_header accepts Meta and VideoBlock
meta = Meta(id='X', title='T', channel='C', duration=60, upload_date='20260101',
            webpage_url='u', captions={'en': 'auto'},
            last_used_at=datetime.now(timezone.utc))
vb = VideoBlock(id='X', title='T', channel='C', url='u', duration=60, upload_date='20260101')
header_m = format_header(meta)
header_v = format_header(vb)
assert header_m == header_v  # shared 4 fields are identical
assert '# T' in header_m and 'Channel: C' in header_m and 'Duration: 1:00' in header_m

print('ok')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()